##### This script is designed to automate the creation of resource entries in eLabFTW, an electronic lab notebook (ELN) and inventory management system. The script extracts data and metadata from a specified CSV file and utilizes the eLabFTW API to create and populate entries with this information.

**Prerequisites:**

- Python installed on your system.
- Required Python libraries: pandas, json, elabapi_python.
- API key for eLabFTW and the API endpoint URL.

##### Steps and Workflow

**Configuration:**

- Set the API host URL and API key for authentication.
- Configure the API client with necessary headers and parameters.

**Reading the CSV File:**

- Load the CSV file containing the resource data into a Pandas DataFrame.
- Inspect and identify the relevant columns needed for metadata extraction.

**Metadata Preparation:**

- Define a base JSON structure for the metadata to be sent to eLabFTW.
- Create a helper function to safely assign values from the DataFrame to the metadata JSON, ensuring that default values are set for any missing or null fields.

**Creating and Updating Resources:**

- Iterate through each row of the DataFrame.
- For each row, create a new resource entry in the specified category.
- Populate the resource metadata with values from the DataFrame.
- Handle any API exceptions that may occur during the creation process.

##### Now we are doing our first step. We will first read the csv and the go further.

In this Cell we are **Reading the CSV File:**

- The script specifies the path to the CSV file containing the resource data.
- It uses the pd.read_csv() function from the Pandas library to read the CSV file into a DataFrame (df).

 **Displaying the DataFrame:**

- The DataFrame is printed to the console, showing the contents of the CSV file.
- A message indicating the contents of the CSV file is printed as well.

**Please replace the 'your_csv_path_here' with your csv file path. Do not delete this sign''. The demo of the path be like ''C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\20240522 .csv'**

In [2]:
import pandas as pd

# Path to the CSV file
csv_path = r'your_csv_path_here'

# Read the CSV file into a DataFrame
df = pd.read_csv(csv_path, encoding='latin-1')


# Display the DataFrame
print("Contents of the CSV file:")
print(df)

Contents of the CSV file:
      CAS number                         Substance name  \
0       108-88-3                                 Toluol   
1        75-05-8                            Acetonitril   
2        50-00-0             Formaldehyd-LÃ¶sung >= 25%   
3     14808-60-7                                  Quarz   
4     14808-60-7                                  Quarz   
..           ...                                    ...   
728  109489-85-2  Barium copper yttrium oxide Ba2Cu3YO6   
729            -   Graphene oxide paste, non-exfoliated   
730   59114-58-8                       Tantalum nitride   
731   59114-58-8                       Tantalum nitride   
732   59114-58-8                       Tantalum nitride   

                                            Title  Location Inventory number  \
0                        0779_C7H8_Toluene_liquid  3201/275    21-523-0138-7   
1                 0271_NH3C2_Acetonitrile_solvent  3201/275    21-523-0014-4   
2                 0622_CH

##### Now we short the csv and save it. Extracting the First 4 Rows:

- The script uses the head() function to extract the first four rows of the DataFrame and assigns them to a new DataFrame (df_first_four_rows).

**Saving to a New CSV File:**

- It specifies the path to save the new CSV file (new_csv_path).
The to_csv() function is utilized to save the DataFrame with the first four rows to a new CSV file, with the index parameter set to False to exclude row indices from the output.

**Displaying the DataFrame:**

- The script prints a message indicating the contents of the CSV file with the first four rows.
- It then prints the DataFrame containing these rows to the console.

**Please replace the 'your_csv_path_here' with your csv file path. Do not delete this sign''. The demo of the path be like ''C:\Users\Master.WR\OneDrive - stud.tu-darmstadt.de\Documents\20240522 .csv'**

In [6]:
# Extract the first 4 rows
df_first_four_rows = df.head(4)

# Path to save the new CSV file
new_csv_path = r'your_csv_path_here'

# Save the DataFrame with the first 4 rows to a new CSV file
df_first_four_rows.to_csv(new_csv_path, index=False)

# Display the DataFrame
print("Contents of the CSV file with the first 4 rows:")
print(df_first_four_rows)

Contents of the CSV file with the first 4 rows:
   CAS number              Substance name                            Title  \
0    108-88-3                      Toluol         0779_C7H8_Toluene_liquid   
1     75-05-8                 Acetonitril  0271_NH3C2_Acetonitrile_solvent   
2     50-00-0  Formaldehyd-LÃ¶sung >= 25%  0622_CH2O_Formaldehyde_solution   
3  14808-60-7                       Quarz           0218_SiO2_Quarz_powder   

   Location Inventory number Structure formula          Manufacturer  \
0  3201/275    21-523-0138-7             C7 H8         Sigma-Aldrich   
1  3201/275    21-523-0014-4          C2 H3 N1            Alfa Aesar   
2  3201/275    22-523-0001-3               NaN  Sigma-Aldrich 252549   
3  3201/275    20-523-0082-4            O2 Si1      Fluka analytical   

  Batch number  Opening date  Purity  
0       244511           NaN     NaN  
1     10223664           NaN     NaN  
2     MKCP3813           NaN     NaN  
3      1393879           NaN     NaN  


##### This script serves as a foundation for automating interactions with eLabFTW using its API. With appropriate configuration and adjustments, it can be extended to perform various tasks such as creating resources in the electronic lab notebook.
**Ensure to replace 'your_api_key_here' with your actual API key obtained from your eLabFTW account.**

In [14]:
import os
import time
import datetime
import csv
import json
import elabapi_python
from elabapi_python.rest import ApiException
import pandas as pd

#########################
#         CONFIG        #
#########################
API_HOST_URL = 'https://eln01-t.ca.hrz.tu-darmstadt.de/api/v2'
# replace with your api key
API_KEY = 'your_api_key_here'
#########################
#      END CONFIG       #
#########################

# Configure the api client
configuration = elabapi_python.Configuration()
configuration.api_key['api_key'] = API_KEY
configuration.api_key_prefix['api_key'] = 'Authorization'
configuration.host = API_HOST_URL
configuration.debug = False
configuration.verify_ssl = False

# Create an instance of the API class
api_client = elabapi_python.ApiClient(configuration)
# Fix issue with Authorization header not being properly set by the generated lib
api_client.set_default_header(header_name='Authorization', header_value=API_KEY)

# Load the items API
items_api = elabapi_python.ItemsApi(api_client)


**Here we are creating the resources. Just change the path of your CSV file. Ensure to replace 'your_csv_path_here' with your actual CSV path and then run the cell.**

In [15]:


# Path to the CSV file
csv_path = r'your_csv_path_here'

# Read the CSV file into a DataFrame
df = pd.read_csv(csv_path, encoding='latin-1')

# Category ID in eLab where the items will be created
category_id = 238  # Adjust this according to your category ID in eLab

# Base JSON for metadata
base_json = '''{
    "elabftw": {
        "display_main_text": false
    },
    "extra_fields": {
        "Inventory number": {
            "type": "text",
            "value": ""
        },
        "Substance name": {
            "type": "text",
            "value": ""
        },
        "Location": {
            "type": "text",
            "value": ""
        },
        "Purity": {
            "type": "text",
            "value": ""
        },
        "Manufacturer": {
            "type": "text",
            "value": ""
        },
        "CAS number": {
            "type": "text",
            "value": ""
        },
        "Batch number": {
            "type": "text",
            "value": ""
        },
        "Opening date": {
            "type": "text",
            "value": ""
        },
        "Structure formula": {
            "type": "text",
            "value": ""
        }
    }
}'''

# Convert base JSON to Python dictionary
metadata_as_json = json.loads(base_json)

# Iterate over the rows in the DataFrame
for index, row in df.iterrows():
    try:
        # Create an item in the correct category
        response = items_api.post_item_with_http_info(body={'category_id': category_id, 'title': row['Title']})
        # Extract the newly created item ID
        location_header_in_response = response[2].get('Location')
        print(f'The newly created item is here: {location_header_in_response}')
        item_id = int(location_header_in_response.split('/').pop())

        # Update metadata with values from the current row
        metadata_as_json = json.loads(base_json)  # Load base JSON for metadata

        # Helper function to safely assign values
        def safe_assign(field_name, row_value, default_value):
            if field_name in metadata_as_json['extra_fields']:
                metadata_as_json['extra_fields'][field_name]['value'] = row_value if pd.notnull(row_value) and row_value.strip() != '' else default_value

        safe_assign('Inventory number', row.get('Inventory number', ''), '')
        safe_assign('Substance name', row.get('Substance name', ''), '')
        safe_assign('Location', row.get('Location', ''), '')
        safe_assign('Purity', row.get('Purity', ''), 'Not specified')
        safe_assign('CAS number', row.get('CAS number', ''), '')
        safe_assign('Batch number', row.get('Batch number', ''), '')
        safe_assign('Manufacturer', row.get('Manufacturer', ''), 'Unknown')
        safe_assign('Structure formula', row.get('Structure formula', ''), 'Not specified')
        safe_assign('Opening date', row.get('Opening date', ''), 'Not specified')

        # Modify the item with the correct metadata
        items_api.patch_item(item_id, body={'title': row['Title'], 'metadata': json.dumps(metadata_as_json)})

        print(f"Item '{row['Title']}' created successfully.")
    except ApiException as e:
        print(f"An error occurred while creating item '{row['Title']}': {e}")


C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


The newly created item is here: https://eln01-t.ca.hrz.tu-darmstadt.de/api/v2/items/7723
Item '0779_C7H8_Toluene_liquid' created successfully.


C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


The newly created item is here: https://eln01-t.ca.hrz.tu-darmstadt.de/api/v2/items/7724
Item '0271_NH3C2_Acetonitrile_solvent' created successfully.


C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


The newly created item is here: https://eln01-t.ca.hrz.tu-darmstadt.de/api/v2/items/7725
Item '0622_CH2O_Formaldehyde_solution' created successfully.


C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'eln01-t.ca.hrz.tu-darmstadt.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


The newly created item is here: https://eln01-t.ca.hrz.tu-darmstadt.de/api/v2/items/7726
Item '0218_SiO2_Quarz_powder' created successfully.
